# NB2 — ScienceWorld State Analysis (REBUILT)

**Purpose:** Establish and validate observable state representations and bounded recovery tests for the irreversibility study.

This notebook builds on the rebuilt NB1 exploration. It distinguishes:

- **Structural object-tree state:** represented object identity, properties and hierarchy, excluding only recursively occurring `temperatureC`.
- **Full object-tree state:** all represented object-tree properties, including temperature.
- **Target-object recovery:** whether the object under study returns to its original identity, properties and location.
- **Environment-level observable recovery:** whether the represented object tree and available task/agent observables return to baseline.

An object-tree fingerprint is an observable-state measure, **not proof of complete simulator-state equality**. Irreversibility claims are bounded by the tested recovery actions and horizon.

## 1. Project setup

Uses the rebuilt project paths:
- Local project: `/content/irrev`
- Main Drive project: `/content/drive/MyDrive/ir`
- Independent backup snapshots: `/content/drive/MyDrive/ir_final`

This notebook does not read legacy `irrev_backup`, `IR`, or `IR_backup` folders.

In [10]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import os, sys, json, hashlib, copy, shutil, time, platform, subprocess
import importlib.metadata as metadata

from google.colab import drive
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/irrev")
DRIVE_ROOT = Path("/content/drive/MyDrive/ir")
BACKUP_ROOT = Path("/content/drive/MyDrive/ir_final")

REPORT_DIR = PROJECT_ROOT / "reports"
DATA_DIR = PROJECT_ROOT / "data"
RESULT_DIR = PROJECT_ROOT / "results" / "scienceworld_state_analysis"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"

for path in (PROJECT_ROOT, DRIVE_ROOT, BACKUP_ROOT, REPORT_DIR, DATA_DIR, RESULT_DIR, NOTEBOOK_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
os.environ["JAVA_TOOL_OPTIONS"] = "-Xlog:os+container=off"

print("Project:", PROJECT_ROOT)
print("Main Drive:", DRIVE_ROOT)
print("Backup Drive:", BACKUP_ROOT)
print("Python:", platform.python_version())
print("Java:")
subprocess.run(["java", "-version"], check=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/irrev
Main Drive: /content/drive/MyDrive/ir
Backup Drive: /content/drive/MyDrive/ir_final
Python: 3.13.18
Java:


CompletedProcess(args=['java', '-version'], returncode=0)

## 2. ScienceWorld installation and environment helpers

The package version is pinned to the version used in NB1. The helper uses the public ScienceWorld API and records actual simulator moves rather than treating Python calls as moves.

In [11]:
try:
    import scienceworld
    from scienceworld import ScienceWorldEnv
    version = getattr(scienceworld, "__version__", "version not exposed")
    print("ScienceWorld already available:", version)
except ImportError:
    %pip install -q scienceworld==1.2.3
    import scienceworld
    from scienceworld import ScienceWorldEnv
    version = getattr(scienceworld, "__version__", "version not exposed")
    print("Installed ScienceWorld:", version)

def make_env(task, variation=0, simplification="easy", step_limit=100):
    env = ScienceWorldEnv(envStepLimit=step_limit)
    env.load(task, variation, simplification)
    observation, info = env.reset()
    return env, observation, info

def safe_close(env):
    try:
        env.close()
    except Exception:
        pass

def get_run_history_safe(env):
    try:
        return env.get_run_history()
    except Exception:
        return None

print("ScienceWorld version:", version)

ScienceWorld already available: 1.2.3
ScienceWorld version: 1.2.3


## 3. State normalisation, fingerprints and object lookup

The structural fingerprint excludes **only** `temperatureC`. Properties such as `stateOfMatter`, `isCombusting`, `hasCombusted`, object UUIDs and hierarchy remain part of the structural representation.

In [12]:
def normalise_tree(value, exclude_temperature=False):
    if isinstance(value, dict):
        return {
            key: normalise_tree(child, exclude_temperature)
            for key, child in sorted(value.items())
            if not (exclude_temperature and key == "temperatureC")
        }
    if isinstance(value, list):
        return [normalise_tree(item, exclude_temperature) for item in value]
    return value

def fingerprint(tree, exclude_temperature=False):
    payload = json.dumps(
        normalise_tree(tree, exclude_temperature),
        sort_keys=True, separators=(",", ":"), ensure_ascii=False
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def structural_fingerprint(tree):
    return fingerprint(tree, exclude_temperature=True)

def full_state_fingerprint(tree):
    return fingerprint(tree, exclude_temperature=False)

def flatten_tree(value, path=""):
    result = {}
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(flatten_tree(child, f"{path}/{key}"))
    elif isinstance(value, list):
        for i, child in enumerate(value):
            result.update(flatten_tree(child, f"{path}[{i}]"))
    else:
        result[path] = value
    return result

def tree_differences(before, after):
    a, b = flatten_tree(before), flatten_tree(after)
    missing = "<MISSING>"
    return [
        {"path": p, "before": a.get(p, missing), "after": b.get(p, missing)}
        for p in sorted(set(a) | set(b))
        if a.get(p, missing) != b.get(p, missing)
    ]

def find_all_objects(tree):
    found = []
    def walk(node, path=""):
        if isinstance(node, dict):
            if "uuid" in node:
                found.append((path, node))
            for key, child in node.items():
                if isinstance(child, (dict, list)):
                    walk(child, f"{path}/{key}")
        elif isinstance(node, list):
            for i, child in enumerate(node):
                walk(child, f"{path}[{i}]")
    walk(tree)
    return found

def find_objects(tree, name=None, uuid=None):
    found = []
    for path, obj in find_all_objects(tree):
        name_match = name is not None and str(obj.get("name", "")).lower() == str(name).lower()
        uuid_match = uuid is not None and str(obj.get("uuid")) == str(uuid)
        if (name is None and uuid is None) or name_match or uuid_match:
            found.append((path, obj))
    return found

def get_object_summary(tree, name=None, uuid=None):
    matches = find_objects(tree, name=name, uuid=uuid)
    return [
        {"path": path, "uuid": str(obj.get("uuid")), "name": obj.get("name"),
         "isDeleted": obj.get("isDeleted"), "object": copy.deepcopy(obj)}
        for path, obj in matches
    ]

def current_action_list(env):
    try:
        return list(env.get_valid_action_object_combinations())
    except Exception:
        return []

def execute_if_valid(env, action):
    valid = current_action_list(env)
    if action not in valid:
        return {"action": action, "executed": False,
                "reason": "Action was not present in the current valid-action list",
                "moves": env.get_num_moves()}
    result = env.step(action)
    return {"action": action, "executed": True, "observation": result[0],
            "reward": result[1], "done": result[2],
            "moves": env.get_num_moves()}

print("Fingerprint and state utilities defined.")

Fingerprint and state utilities defined.


## 4. Fingerprint integrity tests

These are synthetic tests on copied object trees. They verify that the implementation:
- detects UUID changes;
- ignores temperature-only changes in the structural fingerprint;
- retains temperature in the full-state fingerprint;
- detects physical-property and hierarchy changes.

They are software integrity tests, not environment experiments.

In [13]:
env, obs, info = make_env("boil")
baseline_for_tests = copy.deepcopy(env.getObjectTree())
safe_close(env)

objects = find_all_objects(baseline_for_tests)
if not objects:
    raise RuntimeError("No UUID-bearing objects found.")

identity_test = copy.deepcopy(baseline_for_tests)
identity_test_obj = find_all_objects(identity_test)[0][1]
identity_test_obj["uuid"] = "NB2_SYNTHETIC_UUID"
identity_detected = structural_fingerprint(identity_test) != structural_fingerprint(baseline_for_tests)

temperature_test = copy.deepcopy(baseline_for_tests)
temp_obj = next((o for _, o in find_all_objects(temperature_test)
                 if isinstance(o.get("propMaterial"), dict)
                 and o["propMaterial"].get("temperatureC") is not None), None)
if temp_obj is None:
    temperature_exclusion_test = None
    temperature_full_test = None
else:
    temp_obj["propMaterial"]["temperatureC"] += 5.0
    temperature_exclusion_test = structural_fingerprint(temperature_test) == structural_fingerprint(baseline_for_tests)
    temperature_full_test = full_state_fingerprint(temperature_test) != full_state_fingerprint(baseline_for_tests)

physical_test = copy.deepcopy(baseline_for_tests)
physical_obj = next((o for _, o in find_all_objects(physical_test)
                     if isinstance(o.get("propMaterial"), dict)
                     and o["propMaterial"].get("stateOfMatter") is not None), None)
if physical_obj is None:
    physical_change_detected = None
else:
    physical_obj["propMaterial"]["stateOfMatter"] = "NB2_SYNTHETIC_STATE"
    physical_change_detected = structural_fingerprint(physical_test) != structural_fingerprint(baseline_for_tests)

fingerprint_tests = {
    "uuid_change_detected": identity_detected,
    "temperature_only_change_excluded_from_structural": temperature_exclusion_test,
    "temperature_change_included_in_full_state": temperature_full_test,
    "state_of_matter_change_detected": physical_change_detected
}
print(json.dumps(fingerprint_tests, indent=2))
if not identity_detected or temperature_exclusion_test is not True or temperature_full_test is not True or physical_change_detected is not True:
    raise AssertionError("At least one fingerprint integrity test failed.")

{
  "uuid_change_detected": true,
  "temperature_only_change_excluded_from_structural": true,
  "temperature_change_included_in_full_state": true,
  "state_of_matter_change_detected": true
}


## 5. First-step initialisation and environmental drift

Earlier exploratory work reported a first-step structural transition in `boil`, `chemistry-mix` and `freeze` after `look at orange`. NB1 separately established that `look around` was a zero-move observation in its tested `boil` configuration.

This section tests the first-step effect again and records the exact action, validity, move count and fingerprints. It does **not** assume that one observation action is universally free or that one task's behaviour generalises to every task.

In [14]:
initialisation_results = {}
for task in ["boil", "chemistry-mix", "freeze"]:
    env, obs0, info0 = make_env(task)
    reset_tree = copy.deepcopy(env.getObjectTree())
    reset_moves = env.get_num_moves()

    action = "look at orange"
    step_record = execute_if_valid(env, action)
    after_first_tree = copy.deepcopy(env.getObjectTree())

    second_record = execute_if_valid(env, action)
    after_second_tree = copy.deepcopy(env.getObjectTree())

    d01 = tree_differences(reset_tree, after_first_tree)
    d12 = tree_differences(after_first_tree, after_second_tree)

    initialisation_results[task] = {
        "probe_action": action,
        "reset_moves": reset_moves,
        "first_step": step_record,
        "second_step": second_record,
        "reset_to_first_total_differences": len(d01),
        "reset_to_first_non_temperature_differences": sum("temperatureC" not in x["path"] for x in d01),
        "reset_to_first_temperature_differences": sum("temperatureC" in x["path"] for x in d01),
        "first_to_second_total_differences": len(d12),
        "first_to_second_non_temperature_differences": sum("temperatureC" not in x["path"] for x in d12),
        "first_to_second_temperature_differences": sum("temperatureC" in x["path"] for x in d12),
        "reset_structural_fp": structural_fingerprint(reset_tree),
        "first_structural_fp": structural_fingerprint(after_first_tree),
        "second_structural_fp": structural_fingerprint(after_second_tree),
    }
    print("\nTASK:", task)
    print(json.dumps(initialisation_results[task], indent=2))
    safe_close(env)


TASK: boil
{
  "probe_action": "look at orange",
  "reset_moves": 0,
  "first_step": {
    "action": "look at orange",
    "executed": true,
    "observation": "an orange",
    "reward": 0,
    "done": false,
    "moves": 1
  },
  "second_step": {
    "action": "look at orange",
    "executed": true,
    "observation": "an orange",
    "reward": 0,
    "done": false,
    "moves": 2
  },
  "reset_to_first_total_differences": 183,
  "reset_to_first_non_temperature_differences": 102,
  "reset_to_first_temperature_differences": 81,
  "first_to_second_total_differences": 81,
  "first_to_second_non_temperature_differences": 0,
  "first_to_second_temperature_differences": 81,
  "reset_structural_fp": "cf5838238617cd8093b1f6892c8d0c402e3fe673d7efdb95b65b6baeef61b111",
  "first_structural_fp": "ce0025a1f8e01d2e931d56d65a527d4419ef3834e5525e581d5a5a616826d3fb",
  "second_structural_fp": "ce0025a1f8e01d2e931d56d65a527d4419ef3834e5525e581d5a5a616826d3fb"
}

TASK: chemistry-mix
{
  "probe_action":

## 6. Picture recovery — target object versus environment state

NB1 observed a picture relocation sequence and found that the target picture (UUID `17517` in that run) returned to its original parent/location and retained its non-temperature properties. This section repeats a simpler pickup/putdown recovery test from a post-reset baseline and records:
- target object UUID and location;
- target-object properties excluding temperature;
- structural object-tree equality;
- full object-tree equality;
- actual simulator move count.

A structural fingerprint match only covers represented object-tree fields.

In [15]:
env, obs, info = make_env("boil")
picture_baseline = copy.deepcopy(env.getObjectTree())
picture_before = get_object_summary(picture_baseline, name="picture")
if not picture_before:
    raise RuntimeError("No picture object found in the initial boil object tree.")

picture_actions = ["pick up picture", "put down picture"]
picture_action_records = []
for action in picture_actions:
    record = execute_if_valid(env, action)
    picture_action_records.append(record)
    if record.get("done"):
        break

picture_after_tree = copy.deepcopy(env.getObjectTree())
picture_after = get_object_summary(picture_after_tree, name="picture")

def object_location(summary):
    return summary[0]["path"] if summary else None

def object_non_temperature(obj):
    obj = copy.deepcopy(obj)
    material = obj.get("propMaterial")
    if isinstance(material, dict):
        material.pop("temperatureC", None)
    return obj

picture_target_recovery = {
    "actions": picture_action_records,
    "before": picture_before,
    "after": picture_after,
    "same_uuid": bool(picture_before and picture_after and picture_before[0]["uuid"] == picture_after[0]["uuid"]),
    "same_location": object_location(picture_before) == object_location(picture_after),
    "same_non_temperature_properties": bool(
        picture_before and picture_after and
        object_non_temperature(picture_before[0]["object"]) == object_non_temperature(picture_after[0]["object"])
    ),
    "structural_object_tree_recovered": structural_fingerprint(picture_baseline) == structural_fingerprint(picture_after_tree),
    "full_object_tree_recovered": full_state_fingerprint(picture_baseline) == full_state_fingerprint(picture_after_tree),
    "tree_difference_count": len(tree_differences(picture_baseline, picture_after_tree)),
    "moves_after_sequence": env.get_num_moves(),
    "run_history": get_run_history_safe(env)
}
print(json.dumps(picture_target_recovery, indent=2, default=str))
safe_close(env)

{
  "actions": [
    {
      "action": "pick up picture",
      "executed": true,
      "observation": "You move the picture to the inventory.",
      "reward": 0,
      "done": false,
      "moves": 1
    },
    {
      "action": "put down picture",
      "executed": true,
      "observation": "You move the picture to the hallway.",
      "reward": 0,
      "done": false,
      "moves": 2
    }
  ],
  "before": [
    {
      "path": "/contents/hallway-17511/contents/picture-17517",
      "uuid": "17517",
      "name": "picture",
      "isDeleted": false,
      "object": {
        "uuid": "17517",
        "name": "picture",
        "type": "",
        "isDeleted": false,
        "propMaterial": {
          "substanceName": "paper",
          "color": "white",
          "temperatureC": 10.0,
          "thermalConductivity": 0.5,
          "stateOfMatter": "solid",
          "boilingPoint": 10001.0,
          "meltingPoint": 10000.0,
          "isCombusting": false,
          "hasCombust

## 7. Picture relocation and bounded recovery horizons

This experiment attempts the known relocation sequence in `boil`:
1. Move the picture to the art studio.
2. Go to the art studio (H1).
3. Move the picture to the hallway (H2).
4. Go to the hallway (H3).

The notebook records state after each action. A recovery horizon counts recovery actions after the initial relocation. The experiment stops if an action is unavailable or the environment terminates.

In [ ]:
env, obs, info = make_env("boil")
horizon_baseline_tree = copy.deepcopy(env.getObjectTree())
horizon_baseline_fp = structural_fingerprint(horizon_baseline_tree)
horizon_full_baseline_fp = full_state_fingerprint(horizon_baseline_tree)

# Use the action grammar that was actually exercised successfully in rebuilt NB1.
# First relocate the picture to the art studio, then attempt a three-action recovery.
horizon_sequence = [
    ("initial_relocation_1", "pick up picture"),
    ("initial_relocation_2", "go to art studio"),
    ("initial_relocation_3", "put down picture"),
    ("H1", "pick up picture"),
    ("H2", "go to hallway"),
    ("H3", "put down picture"),
]
horizon_records = []
horizon_snapshots = {"baseline": horizon_baseline_tree}
for stage, action in horizon_sequence:
    record = execute_if_valid(env, action)
    record["stage"] = stage
    if record["executed"]:
        tree_now = copy.deepcopy(env.getObjectTree())
        horizon_snapshots[stage] = tree_now
        record["structural_fingerprint"] = structural_fingerprint(tree_now)
        record["full_state_fingerprint"] = full_state_fingerprint(tree_now)
        record["structural_matches_baseline"] = structural_fingerprint(tree_now) == horizon_baseline_fp
        record["full_state_matches_baseline"] = full_state_fingerprint(tree_now) == horizon_full_baseline_fp
    horizon_records.append(record)
    if not record["executed"] or record.get("done"):
        break

horizon_recovery = {
    "steps": horizon_records,
    "horizon_1_structural_recovered": "H1" in horizon_snapshots and structural_fingerprint(horizon_snapshots["H1"]) == horizon_baseline_fp,
    "horizon_2_structural_recovered": "H2" in horizon_snapshots and structural_fingerprint(horizon_snapshots["H2"]) == horizon_baseline_fp,
    "horizon_3_structural_recovered": "H3" in horizon_snapshots and structural_fingerprint(horizon_snapshots["H3"]) == horizon_baseline_fp,
    "horizon_1_full_state_recovered": "H1" in horizon_snapshots and full_state_fingerprint(horizon_snapshots["H1"]) == horizon_full_baseline_fp,
    "horizon_2_full_state_recovered": "H2" in horizon_snapshots and full_state_fingerprint(horizon_snapshots["H2"]) == horizon_full_baseline_fp,
    "horizon_3_full_state_recovered": "H3" in horizon_snapshots and full_state_fingerprint(horizon_snapshots["H3"]) == horizon_full_baseline_fp,
    "note": "H1-H3 are counted after the three-action initial relocation sequence.",
    "final_moves": env.get_num_moves()
}
print(json.dumps(horizon_recovery, indent=2, default=str))
safe_close(env)

## 8. Candidate object-level irreversible action — eat an orange

The orange-consumption test checks whether the original orange UUID remains in the object tree after `eat orange`. This supports only a **candidate object-level irreversibility** finding within the tested state and action space. It does not prove that no possible recovery procedure exists.

In [ ]:
env, obs, info = make_env("boil")
orange_before_tree = copy.deepcopy(env.getObjectTree())
orange_before = get_object_summary(orange_before_tree, name="orange")
orange_action_record = execute_if_valid(env, "eat orange")
orange_after_tree = copy.deepcopy(env.getObjectTree())
orange_after = get_object_summary(orange_after_tree, name="orange")

before_uuids = sorted(x["uuid"] for x in orange_before)
after_uuids = sorted(x["uuid"] for x in orange_after)
orange_consumption = {
    "before": orange_before,
    "action": orange_action_record,
    "after": orange_after,
    "initial_uuids": before_uuids,
    "remaining_uuids": after_uuids,
    "initial_uuids_missing_after": sorted(set(before_uuids) - set(after_uuids)),
    "interpretation": "Candidate object-level irreversible action only; bounded observation does not prove absolute irreversibility."
}
print(json.dumps(orange_consumption, indent=2, default=str))
safe_close(env)

## 9. Freeze-task physical-state validation attempt

The earlier exploration did not demonstrate a successful physical-state transition. This section repeats the limited probes and reports action validity, move counts and object-tree changes. It must not claim that freezing or melting was demonstrated unless a target object's `stateOfMatter` actually changes.

In [ ]:
env, obs, info = make_env("freeze")
freeze_before_tree = copy.deepcopy(env.getObjectTree())
freeze_before_fp = structural_fingerprint(freeze_before_tree)
freeze_actions = ["dunk toilet into cup", "pour cup into toilet"]
freeze_records = []
for action in freeze_actions:
    record = execute_if_valid(env, action)
    freeze_records.append(record)
    if record.get("done"):
        break
freeze_after_tree = copy.deepcopy(env.getObjectTree())

def state_of_matter_by_uuid(tree):
    out = {}
    for path, obj in find_all_objects(tree):
        material = obj.get("propMaterial")
        if isinstance(material, dict) and material.get("stateOfMatter") is not None:
            out[str(obj.get("uuid"))] = {
                "name": obj.get("name"), "path": path,
                "stateOfMatter": material.get("stateOfMatter")
            }
    return out

freeze_before_states = state_of_matter_by_uuid(freeze_before_tree)
freeze_after_states = state_of_matter_by_uuid(freeze_after_tree)
freeze_state_changes = {
    uuid: {"before": freeze_before_states.get(uuid), "after": freeze_after_states.get(uuid)}
    for uuid in sorted(set(freeze_before_states) | set(freeze_after_states))
    if freeze_before_states.get(uuid) != freeze_after_states.get(uuid)
}
freeze_validation = {
    "actions": freeze_records,
    "state_of_matter_changes": freeze_state_changes,
    "structural_fingerprint_changed": structural_fingerprint(freeze_before_tree) != structural_fingerprint(freeze_after_tree),
    "full_tree_difference_count": len(tree_differences(freeze_before_tree, freeze_after_tree)),
    "physical_transition_demonstrated": bool(freeze_state_changes)
}
print(json.dumps(freeze_validation, indent=2, default=str))
safe_close(env)

## 10. Cross-experiment interpretation and methodological decisions

Use these interpretation rules:

- Structural fingerprint equality means equality only for the represented object-tree fields after excluding `temperatureC`.
- Full object-tree fingerprint mismatch can result from temperature drift and does not by itself establish an action-induced structural change.
- A target object returning to its original location is not necessarily full environment recovery.
- Record task/agent observables separately when the API exposes them.
- A valid action that executes does not guarantee a meaningful physical-state transition.
- An invalid action is not evidence of irreversibility; it was not executed.
- A terminal `done=True` flag is not, by itself, proof of irreversibility or task success.
- A finite search over a bounded action set/horizon supports operational labels only; it cannot prove absolute irreversibility.
- The initialisation probe is task- and action-specific; do not assume one universal warm-up action.

In [ ]:
nb2_report = {
    "notebook": "02_scienceworld_state_analysis",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "scienceworld_version": version,
    "fingerprint_tests": fingerprint_tests,
    "initialisation_results": initialisation_results,
    "picture_target_recovery": picture_target_recovery,
    "picture_horizon_recovery": horizon_recovery,
    "orange_consumption_candidate": orange_consumption,
    "freeze_physical_state_validation": freeze_validation,
    "methodological_decisions": [
        "Structural fingerprints exclude only temperatureC.",
        "Full object-tree fingerprints retain temperatureC.",
        "Object-level recovery is distinct from environment-level observable recovery.",
        "Invalid actions are not evidence of irreversibility.",
        "Terminal flags alone do not establish irreversibility or task success.",
        "Finite bounded recovery attempts do not prove absolute irreversibility."
    ]
}
report_path = REPORT_DIR / "NB02_scienceworld_state_analysis_report.json"
report_path.write_text(json.dumps(nb2_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# Store compact experiment artefacts separately for downstream inspection.
summary_path = RESULT_DIR / "NB02_state_analysis_summary.json"
summary_path.write_text(json.dumps(nb2_report, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print("Saved report:", report_path)
print("Saved summary:", summary_path)
print("\nKEY SUMMARY")
print(json.dumps({
    "fingerprint_tests_passed": all(v is True for v in fingerprint_tests.values()),
    "picture_same_uuid": picture_target_recovery["same_uuid"],
    "picture_same_location": picture_target_recovery["same_location"],
    "picture_same_non_temperature_properties": picture_target_recovery["same_non_temperature_properties"],
    "picture_structural_tree_recovered": picture_target_recovery["structural_object_tree_recovered"],
    "horizon_3_structural_recovered": horizon_recovery["horizon_3_structural_recovered"],
    "orange_initial_uuids_missing_after": orange_consumption["initial_uuids_missing_after"],
    "freeze_physical_transition_demonstrated": freeze_validation["physical_transition_demonstrated"]
}, indent=2))

## 11. Output validation

This checks that the report and summary exist and contain the expected top-level experiment sections. It does not certify that any particular scientific result is positive.

In [ ]:
required_outputs = [report_path, summary_path]
for path in required_outputs:
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Missing or empty NB2 output: {path}")

loaded_report = json.loads(report_path.read_text(encoding="utf-8"))
required_sections = [
    "fingerprint_tests", "initialisation_results", "picture_target_recovery",
    "picture_horizon_recovery", "orange_consumption_candidate",
    "freeze_physical_state_validation"
]
missing_sections = [key for key in required_sections if key not in loaded_report]
if missing_sections:
    raise RuntimeError(f"NB2 report is missing sections: {missing_sections}")

print("NB2 output validation passed.")
for path in required_outputs:
    print(path, "|", path.stat().st_size, "bytes")

## 12. Timestamped Drive backup with SHA-256 verification

Run this cell after all experiment, report and validation cells have completed. It copies the current `/content/irrev` project into a new timestamped snapshot under both Drive roots. Existing snapshots are preserved and never overwritten.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_name = f"NB2_snapshot_{timestamp}"

# Save a copy of the notebook only if the user has placed it in the notebooks folder.
# The notebook itself should be saved in Colab before running this backup cell.
source_files = [
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "__pycache__" not in p.parts
    and ".ipynb_checkpoints" not in p.parts
]
if not source_files:
    raise RuntimeError("No project files found to back up.")

backup_results = []
for root in (DRIVE_ROOT, BACKUP_ROOT):
    destination = root / snapshot_name
    if destination.exists():
        raise FileExistsError(f"Refusing to overwrite existing snapshot: {destination}")
    destination.mkdir(parents=True)

    failures = []
    for src in source_files:
        rel = src.relative_to(PROJECT_ROOT)
        dst = destination / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        if src.stat().st_size != dst.stat().st_size or sha256_file(src) != sha256_file(dst):
            failures.append(str(rel))

    backup_results.append({
        "destination": str(destination),
        "file_count": len(source_files),
        "verification_failures": failures
    })

if any(item["verification_failures"] for item in backup_results):
    raise RuntimeError(f"Backup verification failed: {backup_results}")

manifest = {
    "notebook": "02_scienceworld_state_analysis",
    "timestamp_utc": timestamp,
    "snapshot_name": snapshot_name,
    "source_root": str(PROJECT_ROOT),
    "file_count": len(source_files),
    "files": [
        {"path": str(p.relative_to(PROJECT_ROOT)),
         "size_bytes": p.stat().st_size,
         "sha256": sha256_file(p)}
        for p in sorted(source_files)
    ],
    "backup_results": backup_results
}
manifest_path = REPORT_DIR / f"{snapshot_name}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# Also copy the manifest into each snapshot and verify it.
for item in backup_results:
    dst_manifest = Path(item["destination"]) / manifest_path.name
    shutil.copy2(manifest_path, dst_manifest)
    if sha256_file(manifest_path) != sha256_file(dst_manifest):
        raise RuntimeError(f"Manifest verification failed: {dst_manifest}")

print("NB2 BACKUP VERIFIED")
print("Snapshot:", snapshot_name)
for item in backup_results:
    print(item["destination"])
    print("Files verified:", item["file_count"])
    print("Failures:", item["verification_failures"])
print("Manifest:", manifest_path)
print("Manifest SHA-256:", sha256_file(manifest_path))

## 13. NB2 completion checklist

NB2 is ready to close only when:
- all experiment cells execute or any failed/unavailable actions are explicitly recorded;
- fingerprint integrity tests pass;
- the report and summary JSON files are generated and validated;
- both timestamped Drive snapshots pass SHA-256 verification;
- the executed notebook is saved in the project `notebooks` folder or otherwise preserved by the user.

**Scientific caution:** A successful NB2 run establishes tested observable-state behaviour under the documented conditions. It does not establish universal or metaphysical irreversibility.